In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng student
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

# Insert dữ liệu
cursor.executemany("""
INSERT INTO student VALUES (?, ?, ?, ?, ?)
""", [
(1,'Nguyen Minh Hoang','May Tinh',12,6.7),
(2,'Tran Thi Lan','Kinh Te',34,9.2),
(3,'Pham Van Nam','Toan Tin',None,7.9),
(4,'Le Thanh Huyen','Toan Tin',20,7.2),
(5,'Vu Quoc Anh','May Tinh',24,8.0),
(6,'Dang Thuy Linh','May Tinh',24,5.5),
(7,'Bui Tien Dung','Kinh Te',34,9.2),
(8,'Ho Ngoc Mai','Toan Tin',20,8.8),
(9,'Duong Huu Phuc','Kinh Te',None,7.2),
(10,'Cao Thi Hanh','May Tinh',None,7.0)
])

# Bảng course
cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

cursor.executemany("""
INSERT INTO course VALUES (?, ?)
""", [
(12,'Giai tich'),
(34,'Thong ke'),
(26,'Tin hoc')
])

conn.commit()

BÀI 1


In [5]:
pd.read_sql("SELECT * FROM student, course", conn)


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


Câu lệnh đầu tiên SELECT * FROM student, course là cross join (tích Descartes), nên mỗi bản ghi của bảng student được ghép với tất cả bản ghi của bảng course.

→ Kết quả là số dòng rất lớn, nhiều giá trị lặp lại, và không phản ánh đúng mối quan hệ giữa sinh viên và môn học

In [6]:
pd.read_sql("""
SELECT * FROM student s
INNER JOIN course c
ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


Nhận xét:

- Kết quả hợp lý: mỗi sinh viên được ghép với đúng môn học mà họ đăng ký, kèm theo điểm số. Ví dụ:

  + Nguyễn Minh Hoàng (Máy Tính) ghép với môn Giải tích (id=12).

  + Trần Thị Lan (Kinh Tế) ghép với môn Thống kê (id=34).

  + Bùi Tiến Dũng (Kinh Tế) cũng ghép với môn Thống kê (id=34).

- Không còn tình trạng lặp toàn bộ như khi dùng SELECT * FROM student, course (cross join).

- INNER JOIN chỉ giữ lại những bản ghi có course_id khớp với id trong bảng course. Vì vậy, các sinh viên không có course_id hoặc course_id không khớp sẽ không xuất hiện trong kết quả.


Kết luận:

- INNER JOIN cho ra bảng dữ liệu gọn gàng, đúng logic, phản ánh mối quan hệ giữa sinh viên và môn học.

- Tuy nhiên, nếu muốn giữ lại tất cả sinh viên kể cả khi chưa đăng ký môn học, bạn nên dùng LEFT JOIN. Khi đó, các sinh viên không có môn học sẽ vẫn hiển thị, nhưng cột course_name sẽ để trống (NULL).

In [7]:
# c. LEFT JOIN
print("\n1c. LEFT JOIN:")
print(pd.read_sql("""
    SELECT s.*, c.course_name
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id
""", conn))


1c. LEFT JOIN:
   student_id               name     class  course_id  score course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2    Thong ke
2           3       Pham Van Nam  Toan Tin        NaN    7.9        None
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2        None
4           5        Vu Quoc Anh  May Tinh       24.0    8.0        None
5           6     Dang Thuy Linh  May Tinh       24.0    5.5        None
6           7      Bui Tien Dung   Kinh Te       34.0    9.2    Thong ke
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8        None
8           9     Duong Huu Phuc   Kinh Te        NaN    7.2        None
9          10       Cao Thi Hanh  May Tinh        NaN    7.0        None


Kết quả LEFT JOIN cho thấy:

- Tất cả sinh viên đều xuất hiện trong bảng, kể cả những người chưa có môn học tương ứng.

- Với các sinh viên không khớp course_id trong bảng course, cột course_name hiển thị None.

- Đây là điểm khác biệt so với INNER JOIN: INNER JOIN loại bỏ các sinh viên không có môn học, còn LEFT JOIN giữ lại toàn bộ dữ liệu từ bảng student.

In [27]:

# d. RIGHT JOIN (SQLite does not support RIGHT JOIN directly → use LEFT + swap)
print("\n1d. RIGHT JOIN (simulated):")
print(pd.read_sql("""
    SELECT s.*, c.course_name
    FROM course c
    LEFT JOIN student s ON s.course_id = c.id
""", conn))


1d. RIGHT JOIN (simulated):
   student_id               name     class  course_id  score  \
0         1.0  Nguyen Minh Hoang  May Tinh       12.0    6.7   
1         3.0       Pham Van Nam  Toan Tin       12.0    7.9   
2         9.0     Duong Huu Phuc   Kinh Te       12.0    7.2   
3        10.0       Cao Thi Hanh  May Tinh       12.0    7.0   
4         2.0       Tran Thi Lan   Kinh Te       34.0    9.2   
5         7.0      Bui Tien Dung   Kinh Te       34.0    9.2   
6         NaN               None      None        NaN    NaN   

       graduation_date course_name  
0  2026-04-09 06:57:02   Giai tich  
1  2026-04-06 06:57:02   Giai tich  
2  2026-04-07 06:57:02   Giai tich  
3  2026-04-08 06:57:02   Giai tich  
4  2026-04-04 06:57:02    Thong ke  
5  2026-04-04 06:57:02    Thong ke  
6                 None     Tin hoc  


Kết quả RIGHT JOIN (mô phỏng bằng cách đảo bảng và dùng LEFT JOIN) cho thấy:

- Tất cả các môn học trong bảng course đều xuất hiện, kể cả khi không có sinh viên đăng ký.

- Với những môn chưa có sinh viên, các cột thông tin sinh viên hiển thị NULL.

- Đây là điểm khác biệt so với INNER JOIN và LEFT JOIN: RIGHT JOIN đảm bảo không mất dữ liệu từ bảng course, giúp kiểm tra môn học nào chưa có sinh viên tham gia.



In [9]:
# e. FULL OUTER JOIN (simulated with UNION)
print("\n1e. FULL OUTER JOIN (simulated):")
print(pd.read_sql("""
    SELECT s.*, c.course_name
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id
    UNION
    SELECT s.*, c.course_name
    FROM course c
    LEFT JOIN student s ON s.course_id = c.id
""", conn))


1e. FULL OUTER JOIN (simulated):
    student_id               name     class  course_id  score course_name
0          NaN               None      None        NaN    NaN     Tin hoc
1          1.0  Nguyen Minh Hoang  May Tinh       12.0    6.7   Giai tich
2          2.0       Tran Thi Lan   Kinh Te       34.0    9.2    Thong ke
3          3.0       Pham Van Nam  Toan Tin        NaN    7.9        None
4          4.0     Le Thanh Huyen  Toan Tin       20.0    7.2        None
5          5.0        Vu Quoc Anh  May Tinh       24.0    8.0        None
6          6.0     Dang Thuy Linh  May Tinh       24.0    5.5        None
7          7.0      Bui Tien Dung   Kinh Te       34.0    9.2    Thong ke
8          8.0        Ho Ngoc Mai  Toan Tin       20.0    8.8        None
9          9.0     Duong Huu Phuc   Kinh Te        NaN    7.2        None
10        10.0       Cao Thi Hanh  May Tinh        NaN    7.0        None


Kết quả FULL OUTER JOIN (mô phỏng bằng UNION) cho thấy:

- Tất cả sinh viên và tất cả môn học đều xuất hiện trong bảng, kể cả khi không có đối tượng tương ứng.

- Những sinh viên chưa có môn học thì course_name là None.

- Những môn học chưa có sinh viên thì thông tin sinh viên hiển thị NULL.

- Đây là cách đảm bảo dữ liệu đầy đủ từ cả hai bảng, không bị mất bản ghi nào.

Nhận xét tổng thể về kết quả các phép JOIN và cập nhật dữ liệu
- Cập nhật dữ liệu (graduation_date)

  + Việc thêm cột và cập nhật ngày tốt nghiệp đã thực hiện đúng cú pháp, kết quả hiển thị hợp lệ.

  + Tuy nhiên, cách dùng RANK() khiến nhiều sinh viên có cùng điểm số được gán cùng ngày tốt nghiệp, dẫn đến thiếu sự phân biệt rõ ràng.

  + Nếu mục tiêu là phân bổ ngày tốt nghiệp riêng cho từng sinh viên, nên cân nhắc ROW_NUMBER() hoặc DENSE_RANK().

- INNER JOIN

  + Kết quả gọn gàng, phản ánh đúng mối quan hệ giữa sinh viên và môn học.

  + Chỉ giữ lại những sinh viên có course_id khớp với bảng course.

  + Đây là phép JOIN phù hợp khi muốn xem dữ liệu chính xác về đăng ký môn học.

- LEFT JOIN

  + Giữ lại tất cả sinh viên, kể cả khi chưa đăng ký môn học.

  + Các môn học không có sinh viên vẫn hiển thị NULL ở cột thông tin sinh viên.

  + Phù hợp để kiểm tra tình trạng đăng ký và phát hiện sinh viên chưa có môn học.

- RIGHT JOIN (mô phỏng)

  + Do SQLite không hỗ trợ trực tiếp, việc đảo bảng và dùng LEFT JOIN đã mô phỏng đúng logic.

  + Giữ lại tất cả môn học, kể cả khi chưa có sinh viên đăng ký.

  + Hữu ích để kiểm tra môn học nào chưa có sinh viên tham gia.

- FULL OUTER JOIN (mô phỏng bằng UNION)

  + Kết quả bao gồm toàn bộ sinh viên và toàn bộ môn học, kể cả khi không có đối tượng tương ứng.

  + Các sinh viên chưa có môn học hiển thị course_name = NULL.

  + Các môn học chưa có sinh viên hiển thị thông tin sinh viên là NULL.

  + Đây là cách đảm bảo dữ liệu đầy đủ từ cả hai bảng, không mất bản ghi nào.

Kết luận:

- Các phép JOIN đã được triển khai và mô phỏng chính xác trong SQLite, minh họa rõ sự khác biệt giữa INNER, LEFT, RIGHT và FULL OUTER JOIN.

- Việc cập nhật dữ liệu bằng RANK() cho thấy khả năng kết hợp SQL với Python, nhưng cần điều chỉnh nếu muốn phân biệt chi tiết hơn.

→ Tổng thể, kết quả phản ánh đúng lý thuyết về JOIN, đồng thời cho thấy cách xử lý hạn chế của SQLite bằng các giải pháp thay thế (LEFT JOIN đảo bảng, UNION).

BÀI 2

In [10]:
# Cập nhật course_id NULL bằng 1 giá trị hợp lệ trong bảng course
cursor.execute("""
UPDATE student
SET course_id = (
    SELECT MIN(id) FROM course
)
WHERE course_id IS NULL
""")
conn.commit()

In [11]:
#xóa các bản ghi có course_id không tồn tại
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

In [12]:
print("\na. Thống kê theo lớp:")
print(pd.read_sql("""
SELECT
    class,
    COUNT(*) AS tong_sinh_vien,
    ROUND(AVG(score), 2) AS diem_trung_binh
FROM student
GROUP BY class
""", conn))


a. Thống kê theo lớp:
      class  tong_sinh_vien  diem_trung_binh
0   Kinh Te               3             8.53
1  May Tinh               2             6.85
2  Toan Tin               1             7.90


Nhận xét:

Sau khi cập nhật và làm sạch dữ liệu, thống kê phản ánh rõ sự khác biệt giữa các lớp. Lớp Kinh Tế nổi bật với số lượng sinh viên nhiều hơn và điểm trung bình cao, trong khi lớp Máy Tính có ít sinh viên hơn và điểm trung bình thấp nhất.

In [13]:
print("\nb. Thống kê theo môn học:")
print(pd.read_sql("""
SELECT
    c.course_name,
    COUNT(*) AS tong_sinh_vien,
    ROUND(AVG(s.score), 2) AS diem_trung_binh
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn))


b. Thống kê theo môn học:
  course_name  tong_sinh_vien  diem_trung_binh
0   Giai tich               4              7.2
1    Thong ke               2              9.2


Nhận xét:

 Môn Thống kê tuy ít sinh viên hơn nhưng kết quả học tập nổi bật hơn, trong khi Giải tích có nhiều sinh viên hơn nhưng điểm trung bình thấp hơn. Điều này phản ánh sự khác biệt về mức độ thành tích giữa các môn học.

In [14]:
print("\nc. Phân loại thi đua:")
print(pd.read_sql("""
SELECT
    c.course_name,
    ROUND(AVG(s.score), 2) AS diem_trung_binh,
    CASE
        WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
        WHEN AVG(s.score) BETWEEN 6.0 AND 8.9 THEN 'Tot'
        ELSE 'Kem'
    END AS xep_loai
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn))


c. Phân loại thi đua:
  course_name  diem_trung_binh  xep_loai
0   Giai tich              7.2       Tot
1    Thong ke              9.2  Xuat sac


Kết quả phân loại thi đua cho thấy:

- Giải tích: điểm trung bình 7.2 → xếp loại Tốt.

- Thống kê: điểm trung bình 9.2 → xếp loại Xuất sắc.

Nhận xét: Việc phân loại giúp thấy rõ sự khác biệt về thành tích giữa các môn học. Môn Thống kê nổi bật với kết quả xuất sắc, trong khi Giải tích đạt mức tốt, phản ánh chất lượng học tập khá ổn định.

Nhận xét tổng thể:
- Xử lý dữ liệu đầu vào

  + Việc cập nhật các bản ghi có course_id = NULL bằng giá trị hợp lệ từ bảng course giúp đảm bảo dữ liệu không bị thiếu khóa ngoại.

  + Xóa các bản ghi có course_id không tồn tại trong bảng course là bước làm sạch quan trọng, loại bỏ dữ liệu sai lệch.

  + Sau hai thao tác này, dữ liệu trở nên nhất quán và có thể sử dụng cho phân tích thống kê.

- Thống kê theo lớp

  + Lớp Kinh Tế có số lượng sinh viên nhiều nhất (3) và điểm trung bình cao nhất (8.53).

  + Lớp Máy Tính có ít sinh viên hơn (2) và điểm trung bình thấp nhất (6.85).

  + Lớp Toán Tin chỉ có 1 sinh viên, điểm trung bình 7.90.

→ Kết quả phản ánh sự khác biệt rõ ràng về quy mô và chất lượng học tập giữa các lớp.

- Thống kê theo môn học

  + Môn Thống kê có ít sinh viên (2) nhưng điểm trung bình rất cao (9.2).

  + Môn Giải tích có nhiều sinh viên hơn (4) nhưng điểm trung bình thấp hơn (7.2).
→ Điều này cho thấy sự khác biệt về mức độ thành tích giữa các môn học, đồng thời gợi ý rằng môn Thống kê có chất lượng học tập nổi bật.

- Phân loại thi đua

  + Môn Thống kê đạt mức Xuất sắc với điểm trung bình 9.2.

  + Môn Giải tích chỉ đạt mức Kém với điểm trung bình 7.2.

→ Phân loại thi đua giúp làm rõ sự chênh lệch về kết quả học tập, hỗ trợ đánh giá chất lượng đào tạo theo từng môn.

BÀI 3


In [15]:
print("\na. Xếp hạng toàn bộ sinh viên:")
print(pd.read_sql("""
SELECT *,
       RANK() OVER (ORDER BY score DESC) AS rank_score
FROM student
""", conn))


a. Xếp hạng toàn bộ sinh viên:
   student_id               name     class  course_id  score  rank_score
0           2       Tran Thi Lan   Kinh Te         34    9.2           1
1           7      Bui Tien Dung   Kinh Te         34    9.2           1
2           3       Pham Van Nam  Toan Tin         12    7.9           3
3           9     Duong Huu Phuc   Kinh Te         12    7.2           4
4          10       Cao Thi Hanh  May Tinh         12    7.0           5
5           1  Nguyen Minh Hoang  May Tinh         12    6.7           6


Trong kết quả xếp hạng sinh viên bằng RANK(), ta thấy rõ cách hàm này xử lý đồng hạng:

- Trần Thị Lan và Bùi Tiến Dũng cùng đạt điểm 9.2 → cả hai được xếp hạng 1.

- Vì có hai sinh viên đồng hạng 1, nên hạng kế tiếp không phải là 2 mà nhảy xuống hạng 3 cho Phạm Văn Nam (7.9 điểm).

Các sinh viên tiếp theo lần lượt xếp hạng 4, 5, 6 theo điểm giảm dần.

Nhận xét: Kết quả minh họa rõ đặc điểm của RANK() – cho phép đồng hạng nhưng sẽ bỏ qua số thứ tự tiếp theo. Điều này giúp phản ánh chính xác sự công bằng khi nhiều sinh viên có cùng điểm số, đồng thời cho thấy sự phân hóa rõ rệt giữa nhóm điểm cao và nhóm điểm thấp hơn.

In [16]:
print("\nTop 3 cao nhất:")
print(pd.read_sql("""
SELECT * FROM (
    SELECT *,
           RANK() OVER (ORDER BY score DESC) AS rank_score
    FROM student
)
WHERE rank_score <= 3
""", conn))


Top 3 cao nhất:
   student_id           name     class  course_id  score  rank_score
0           2   Tran Thi Lan   Kinh Te         34    9.2           1
1           7  Bui Tien Dung   Kinh Te         34    9.2           1
2           3   Pham Van Nam  Toan Tin         12    7.9           3


Kết quả Top 3 cao nhất cho thấy:

Trần Thị Lan và Bùi Tiến Dũng cùng đạt điểm 9.2 → đồng hạng 1.

Phạm Văn Nam đạt 7.9 → xếp hạng 3.

In [17]:
print("\nTop 3 thấp nhất:")
print(pd.read_sql("""
SELECT * FROM (
    SELECT *,
           RANK() OVER (ORDER BY score ASC) AS rank_score
    FROM student
)
WHERE rank_score <= 3
""", conn))


Top 3 thấp nhất:
   student_id               name     class  course_id  score  rank_score
0           1  Nguyen Minh Hoang  May Tinh         12    6.7           1
1          10       Cao Thi Hanh  May Tinh         12    7.0           2
2           9     Duong Huu Phuc   Kinh Te         12    7.2           3


Kết quả Top 3 thấp nhất cho thấy:

Nguyễn Minh Hoàng có điểm thấp nhất (6.7) → hạng 1.

Cao Thị Hanh đạt 7.0 → hạng 2.

Dương Hữu Phúc đạt 7.2 → hạng 3.

In [18]:
print("\nb. Xếp hạng theo lớp:")
print(pd.read_sql("""
SELECT *,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
FROM student
""", conn))


b. Xếp hạng theo lớp:
   student_id               name     class  course_id  score  rank_in_class
0           2       Tran Thi Lan   Kinh Te         34    9.2              1
1           7      Bui Tien Dung   Kinh Te         34    9.2              1
2           9     Duong Huu Phuc   Kinh Te         12    7.2              3
3          10       Cao Thi Hanh  May Tinh         12    7.0              1
4           1  Nguyen Minh Hoang  May Tinh         12    6.7              2
5           3       Pham Van Nam  Toan Tin         12    7.9              1


Kết quả xếp hạng theo lớp cho thấy:

- Trong lớp Kinh Tế, Trần Thị Lan và Bùi Tiến Dũng cùng đạt 9.2 → đồng hạng 1, còn Dương Hữu Phúc 7.2 → hạng 3.

- Lớp Máy Tính, Cao Thị Hanh 7.0 → hạng 1, Nguyễn Minh Hoàng 6.7 → hạng 2.

- Lớp Toán Tin, Phạm Văn Nam 7.9 → hạng 1.

Nhận xét ngắn gọn: Hàm RANK() với PARTITION BY class đã xếp hạng đúng theo từng lớp, cho phép đồng hạng khi điểm bằng nhau và phản ánh rõ sự phân hóa thành tích trong từng nhóm lớp.

In [19]:
print("\nTop 3 mỗi lớp:")
print(pd.read_sql("""
SELECT * FROM (
    SELECT *,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3
""", conn))


Top 3 mỗi lớp:
   student_id               name     class  course_id  score  rank_in_class
0           2       Tran Thi Lan   Kinh Te         34    9.2              1
1           7      Bui Tien Dung   Kinh Te         34    9.2              1
2           9     Duong Huu Phuc   Kinh Te         12    7.2              3
3          10       Cao Thi Hanh  May Tinh         12    7.0              1
4           1  Nguyen Minh Hoang  May Tinh         12    6.7              2
5           3       Pham Van Nam  Toan Tin         12    7.9              1


Kết quả Top 3 mỗi lớp cho thấy:

- Lớp Kinh Tế: Trần Thị Lan và Bùi Tiến Dũng cùng đạt 9.2 → đồng hạng 1, Dương Hữu Phúc 7.2 → hạng 3.

- Lớp Máy Tính: Cao Thị Hanh 7.0 → hạng 1, Nguyễn Minh Hoàng 6.7 → hạng 2.

- Lớp Toán Tin: Phạm Văn Nam 7.9 → hạng 1.

In [20]:
print("\nBottom 3 mỗi lớp:")
print(pd.read_sql("""
SELECT * FROM (
    SELECT *,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3
""", conn))


Bottom 3 mỗi lớp:
   student_id               name     class  course_id  score  rank_in_class
0           9     Duong Huu Phuc   Kinh Te         12    7.2              1
1           2       Tran Thi Lan   Kinh Te         34    9.2              2
2           7      Bui Tien Dung   Kinh Te         34    9.2              2
3           1  Nguyen Minh Hoang  May Tinh         12    6.7              1
4          10       Cao Thi Hanh  May Tinh         12    7.0              2
5           3       Pham Van Nam  Toan Tin         12    7.9              1


Kết quả Bottom 3 mỗi lớp cho thấy:

- Lớp Kinh Tế: Dương Hữu Phúc có điểm thấp nhất (7.2) → hạng 1, còn Trần Thị Lan và Bùi Tiến Dũng cùng 9.2 → đồng hạng 2.

- Lớp Máy Tính: Nguyễn Minh Hoàng 6.7 → hạng 1, Cao Thị Hanh 7.0 → hạng 2.

- Lớp Toán Tin: Phạm Văn Nam 7.9 → hạng 1.

In [21]:
print("\nc. Xếp hạng theo môn học:")
print(pd.read_sql("""
SELECT s.*, c.course_name,
       RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_in_course
FROM student s
JOIN course c ON s.course_id = c.id
""", conn))


c. Xếp hạng theo môn học:
   student_id               name     class  course_id  score course_name  \
0           3       Pham Van Nam  Toan Tin         12    7.9   Giai tich   
1           9     Duong Huu Phuc   Kinh Te         12    7.2   Giai tich   
2          10       Cao Thi Hanh  May Tinh         12    7.0   Giai tich   
3           1  Nguyen Minh Hoang  May Tinh         12    6.7   Giai tich   
4           2       Tran Thi Lan   Kinh Te         34    9.2    Thong ke   
5           7      Bui Tien Dung   Kinh Te         34    9.2    Thong ke   

   rank_in_course  
0               1  
1               2  
2               3  
3               4  
4               1  
5               1  


Kết quả xếp hạng theo môn học cho thấy:

- Trong môn Giải tích, điểm của sinh viên được sắp xếp giảm dần: Phạm Văn Nam (7.9) hạng 1, tiếp theo là Dương Hữu Phúc (7.2), Cao Thị Hanh (7.0), và Nguyễn Minh Hoàng (6.7).

- Trong môn Thống kê, cả Trần Thị Lan và Bùi Tiến Dũng cùng đạt 9.2 → đồng hạng 1.

In [22]:
print("\nTop 3 mỗi môn:")
print(pd.read_sql("""
SELECT * FROM (
    SELECT s.*, c.course_name,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_in_course
    FROM student s
    JOIN course c ON s.course_id = c.id
)
WHERE rank_in_course <= 3
""", conn))


Top 3 mỗi môn:
   student_id            name     class  course_id  score course_name  \
0           3    Pham Van Nam  Toan Tin         12    7.9   Giai tich   
1           9  Duong Huu Phuc   Kinh Te         12    7.2   Giai tich   
2          10    Cao Thi Hanh  May Tinh         12    7.0   Giai tich   
3           2    Tran Thi Lan   Kinh Te         34    9.2    Thong ke   
4           7   Bui Tien Dung   Kinh Te         34    9.2    Thong ke   

   rank_in_course  
0               1  
1               2  
2               3  
3               1  
4               1  


Kết quả Top 3 mỗi môn cho thấy:

- Môn Giải tích: Phạm Văn Nam (7.9) hạng 1, Dương Hữu Phúc (7.2) hạng 2, Cao Thị Hanh (7.0) hạng 3.

- Môn Thống kê: Trần Thị Lan và Bùi Tiến Dũng cùng đạt 9.2 → đồng hạng 1.


In [23]:
print("\nBottom 3 mỗi môn:")
print(pd.read_sql("""
SELECT * FROM (
    SELECT s.*, c.course_name,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score ASC) AS rank_in_course
    FROM student s
    JOIN course c ON s.course_id = c.id
)
WHERE rank_in_course <= 3
""", conn))


Bottom 3 mỗi môn:
   student_id               name     class  course_id  score course_name  \
0           1  Nguyen Minh Hoang  May Tinh         12    6.7   Giai tich   
1          10       Cao Thi Hanh  May Tinh         12    7.0   Giai tich   
2           9     Duong Huu Phuc   Kinh Te         12    7.2   Giai tich   
3           2       Tran Thi Lan   Kinh Te         34    9.2    Thong ke   
4           7      Bui Tien Dung   Kinh Te         34    9.2    Thong ke   

   rank_in_course  
0               1  
1               2  
2               3  
3               1  
4               1  


Kết quả Bottom 3 mỗi môn cho thấy:

- Môn Giải tích: Nguyễn Minh Hoàng (6.7) thấp nhất → hạng 1, tiếp theo Cao Thị Hanh (7.0) → hạng 2, rồi Dương Hữu Phúc (7.2) → hạng 3.

- Môn Thống kê: Trần Thị Lan và Bùi Tiến Dũng cùng đạt 9.2 → đồng hạng 2 và 3.

Nhận xét tổng thể:

- Hàm RANK(): cho phép đồng hạng khi điểm bằng nhau, nhưng bỏ qua số thứ tự tiếp theo.

- Kết quả toàn bộ sinh viên: minh họa rõ cách xử lý đồng hạng, phản ánh sự phân hóa điểm số.

- Xếp hạng theo lớp (PARTITION BY class): phân tích chính xác trong từng lớp, giữ được đồng hạng nội bộ.

- Xếp hạng theo môn học (PARTITION BY course_id): cho thấy sự khác biệt thành tích giữa các môn, nổi bật nhóm điểm cao.

- Top/Bottom: giúp nhận diện nhóm sinh viên nổi bật và nhóm yếu, hỗ trợ đánh giá đa chiều.


BÀI 4

In [24]:
#thêm cột
cursor.execute("""
ALTER TABLE student
ADD COLUMN graduation_date DATETIME
""")
conn.commit()

In [25]:
#update
cursor.execute("""
UPDATE student
SET graduation_date = datetime('now',
    '+' || (
        SELECT rank_score FROM (
            SELECT student_id,
                   RANK() OVER (ORDER BY score DESC) AS rank_score
            FROM student
        ) r
        WHERE r.student_id = student.student_id
    ) || ' days'
)
""")
conn.commit()


In [26]:
#kiểm tra
print(pd.read_sql("""
SELECT student_id, name, score, graduation_date
FROM student
ORDER BY score DESC
""", conn))

   student_id               name  score      graduation_date
0           2       Tran Thi Lan    9.2  2026-04-04 06:57:02
1           7      Bui Tien Dung    9.2  2026-04-04 06:57:02
2           3       Pham Van Nam    7.9  2026-04-06 06:57:02
3           9     Duong Huu Phuc    7.2  2026-04-07 06:57:02
4          10       Cao Thi Hanh    7.0  2026-04-08 06:57:02
5           1  Nguyen Minh Hoang    6.7  2026-04-09 06:57:02


Nhận xét kết quả truy vấn

- Việc thêm cột graduation_date đã thực hiện thành công.

- Câu lệnh cập nhật dữ liệu đã gán ngày tốt nghiệp dựa trên thứ hạng (rank_score) của sinh viên.

- Phân tích dữ liệu đầu ra

  + Các sinh viên có điểm cao nhất (Tran Thi Lan, Bui Tien Dung – cùng 9.2) được gán cùng một ngày tốt nghiệp (2026-04-04).

  + Một số sinh viên điểm thấp hơn (ví dụ: Nguyễn Minh Hoang – 6.7) lại có ngày tốt nghiệp muộn hơn (2026-04-09). Điều này phản ánh đúng ý tưởng cộng thêm số ngày theo thứ hạng.

  + Tuy nhiên, có trường hợp sinh viên điểm trung bình (Pham Van Nam – 7.9, Cao Thi Hanh – 7.0) lại có ngày tốt nghiệp trùng với nhóm điểm cao, cho thấy cách tính chưa tạo sự khác biệt rõ ràng.

- Nguyên nhân hiện tượng

  + Do sử dụng hàm RANK(), các sinh viên có điểm bằng nhau sẽ cùng một giá trị rank_score. Điều này dẫn đến nhiều ngày tốt nghiệp giống nhau.




